# Tracing in OpenGGCM fields

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import pyvista as pv
import xarray as xr
from scipy import constants

import ggcmpy.tracing


def to_mesh_lines(df):
    positions = df[["x", "y", "z"]].to_numpy()
    mesh = pv.PolyData(positions)
    lines = pv.lines_from_points(positions)
    return mesh, lines


def plot_trajectory(plotter, df, **kwargs):
    _, lines = to_mesh_lines(df)
    plotter.add_mesh(lines, **kwargs)

### Load an actual OpenGGCM dataset

It's not from an actually meaningful simulation, but it'll do for now. The code below loads the data (which was generated with `etajout=true`), and then rescales to base SI units (FIXME: we should have the option to just trace in the original units).

In addition, it sets the electric field to zero to make things simpler.

In [ ]:
R_E = 6.371e6  # m

ds = xr.open_dataset(ggcmpy.sample_dir / "test0008.3df.000007")
x, y, z = R_E * ds.x.to_numpy(), R_E * ds.y.to_numpy(), R_E * ds.z.to_numpy()
ds = ds.assign_coords(
    x=("x", x),
    y=("y", y),
    z=("z", z),
    x_nc=("x_nc", 0.5 * (x[1:] + x[:-1])),
    y_nc=("y_nc", 0.5 * (y[1:] + y[:-1])),
    z_nc=("z_nc", 0.5 * (z[1:] + z[:-1])),
)
ds["bx1"] = (("x_nc", "y", "z"), 1e-9 * ds.bx1.to_numpy()[:-1, :, :])
ds["by1"] = (("x", "y_nc", "z"), 1e-9 * ds.by1.to_numpy()[:, :-1, :])
ds["bz1"] = (("x", "y", "z_nc"), 1e-9 * ds.bz1.to_numpy()[:, :, :-1])
ds["eflx"] = (("x", "y_nc", "z_nc"), 0 * ds.eflx.to_numpy()[:, :-1, :-1])
ds["efly"] = (("x_nc", "y", "z_nc"), 0 * ds.efly.to_numpy()[:-1, :, :-1])
ds["eflz"] = (("x_nc", "y_nc", "z"), 0 * ds.eflz.to_numpy()[:-1, :-1, :])
ds

### Trace using the Boris integrator

That's pretty much just the same as in the dipole example, but using the OpenGGCM fields.

In [ ]:
q, m = -constants.e, constants.m_e
boris = ggcmpy.tracing.integrator.boris_cxx(ds, q=q, m=m)
x0 = np.array([-5.0 * R_E, 0, 0])
B_x0 = boris._fields.B(x0)

E_kin = 10.0 * 1e6 * constants.e  # 10 MeV in J
gamma = 1.0 + E_kin / (m * constants.c**2)
v_e = constants.c * np.sqrt(1.0 - 1.0 / gamma**2)

v0 = np.array([0.0, v_e / np.sqrt(2.0), v_e / np.sqrt(2.0)])  # [m/s]
u0 = gamma * v0 / constants.c
prts = pd.DataFrame(
    [[0, 0.0, *x0, *u0]], columns=["id", "time", "x", "y", "z", "ux", "uy", "uz"]
)

om_ce = np.abs(q) * np.linalg.norm(B_x0) / (gamma * m)  # gyrofrequency
r_ce = m * v_e / (q * np.linalg.norm(B_x0))  # gyroradius

t_final = 500 * 2 * np.pi / om_ce  # [s]
print(f"B={B_x0} om_ce={om_ce} r_ce={r_ce} t_final={t_final}")  # noqa: T201

df = boris.integrate(prts, t_final=t_final, dt_max_gyro=0.1, snapshot_interval_steps=1)
df

In [ ]:
import scipy.integrate


def trace_field_line(r0, get_B, r_min=0.5, r_max=10.0):
    """Trace a magnetic field line starting at position r0 in the field
    provided by the get_B function.

    This function integrates the field line in the positive direction
    until it get either closer than `r_min` to the origin, or outside of `r_max`.

    Returns a pandas DataFrame with columns 'time', 'x', 'y', 'z'.
    """

    def rhs(t, x):  # noqa: ARG001
        B = get_B(x)
        B /= np.linalg.norm(B)
        return B

    def outside(t, x):  # noqa: ARG001
        return np.linalg.norm(x) - r_max  # stop if r > r_max

    outside.terminal = True
    outside.direction = 1  # only trigger when approaching from inside

    def inside(t, x):  # noqa: ARG001
        return np.linalg.norm(x) - r_min  # stop if r < r_min

    inside.terminal = True
    inside.direction = -1  # only trigger when approaching from outside

    tf = 1e9
    sol = scipy.integrate.solve_ivp(
        rhs, (0.0, tf), r0, events=[outside, inside], max_step=0.1 * R_E
    )
    sol2 = scipy.integrate.solve_ivp(
        rhs, (tf, 0.0), r0, events=[outside, inside], max_step=0.1 * R_E
    )
    dfs = [
        pd.DataFrame(np.column_stack((s.t, s.y.T)), columns=["time", "x", "y", "z"])
        for s in [sol, sol2]
    ]
    return pd.concat((dfs[1].iloc[::-1], dfs[0]), ignore_index=True)


def trace_field_lines(seeds, B_func, integrate_kwargs):
    """Trace multiple field lines.

    This is a convenience wrapper around trace_field_line.
    `seeds` is a list of starting positions.
    `B_func` is the magnetic field function.
    `integrate_kwargs` is a dictionary of additional keyword arguments to pass to
    `trace_field_line`.

    Returns a list of pandas DataFrames, one per seed point."""
    return [trace_field_line(r0, B_func, **integrate_kwargs) for r0 in seeds]


seeds = [
    r * np.array([np.cos(phi), np.sin(phi), 0.0])
    for phi in np.linspace(0.0, 2.0 * np.pi, 16, endpoint=False)
    for r in [3.0 * R_E, 5.0 * R_E]
]

dfs = trace_field_lines(
    seeds,
    boris._fields.B,
    integrate_kwargs={"r_min": 2 * R_E, "r_max": 20.0 * R_E},
)

### Plot the trajectory

Not all that exciting, but it looks reasonable...

In [ ]:
plotter = pv.Plotter()
plot_trajectory(plotter, df, line_width=2, color="red")
for _df in dfs:
    plot_trajectory(plotter, _df, line_width=1, color="grey")
plotter.show()

### Check energy conservation

Looks good...

In [ ]:
df["E"] = (
    np.sqrt(1.0 + np.linalg.norm(df[["ux", "uy", "uz"]].values, axis=1) ** 2) - 1.0
) * (m * constants.c**2)
df.plot(x="time", y="E", title="Kinetic energy of the particle over time");